# CO2 refrigeration system 

This notebook models a CO2 transcritical booster refrigeration system of the kind used
in supermarkets.
The model is built with [TESPy](https://tespy.readthedocs.io/) (Thermal Engineering Simulation
in Python) and [CoolProp](http://www.coolprop.org/) for the R744 (CO2) property data.

The booster-cycle topology, component set, and boudnary conditions used
here (evaporator pressures and cooling loads, compressor isentropic efficiencies, gas cooler
pressure and outlet temperature, receiver pressure) are taken from:

> Francesco D’Ettorre, Christian Heerup, *"Chapter 5 - Modelling of a commercial refrigeration system in Python"*,
> Case Studies in Energy Systems, Elsevier, 2026, Pages 163-189, ISBN 9780443238635, https://doi.org/10.1016/B978-0-443-23863-5.00010-5.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import CoolProp.CoolProp as CP

from tespy.networks import Network
from tespy.tools.helpers import merge_dicts
from tespy.components import (
    Sink, Source, SimpleHeatExchanger, CycleCloser, Valve, Compressor,
    DropletSeparator, Merge, Splitter, SectionedHeatExchanger
)
from tespy.connections import Connection, Ref
from tespy.tools import get_plotting_data
from fluprodia import FluidPropertyDiagram

## ModelTemplate

Base class providing the parameter lookup / get / set plumbing and default plotting
hooks shared by all models.

In [ ]:
DIAGRAMS = {}

class ModelTemplate():

    def __init__(self) -> None:
        self.parameter_lookup = self._parameter_lookup()
        self._create_network()

    def _create_network(self) -> None:
        self.nw = Network()
        self.nw.units.set_defaults(
            **{"temperature": "°C", "pressure": "bar"}
        )

    def _parameter_lookup(self) -> dict:
        return {}

    def _map_parameter(self, parameter: str) -> tuple:
        return self.parameter_lookup[parameter]

    def _map_to_input_dict(self, **kwargs) -> dict:
        input_dict = {}
        for param, value in kwargs.items():
            if param not in self.parameter_lookup:
                msg = (
                    f"The parameter {param} is not mapped to any input of the "
                    "model. The following parameters are available:\n"
                    f"{', '.join(self.parameter_lookup)}."
                )
                raise KeyError(msg)
            key = self._map_parameter(param)
            input_dict = merge_dicts(
                input_dict,
                {key[0]: {key[1]: {key[2]: value}}}
            )
        return input_dict

    def get_parameter(self, parameter: str) -> float:
        mapped = self._map_parameter(parameter)
        if mapped[0] == "Connections":
            return self.nw.get_conn(mapped[1]).get_attr(mapped[2]).val

        elif mapped[0] == "Components":
            return self.nw.get_comp(mapped[1]).get_attr(mapped[2]).val

    def set_parameters(self, **kwargs) -> None:
        input_dict = self._map_to_input_dict(**kwargs)
        if "Connections" in input_dict:
            for c, params in input_dict["Connections"].items():
                self.nw.get_conn(c).set_attr(**params)

        if "Components" in input_dict:
            for c, params in input_dict["Components"].items():
                self.nw.get_comp(c).set_attr(**params)

    def solve_model(self, **kwargs) -> None:
        self.set_parameters(**kwargs)

        self._solved = False
        self.nw.solve("design")

        if self.nw.status == 0:
            self._solved = True
        # is not required in this example, but could lead to handling some
        # stuff
        elif self.nw.status == 1:
            self._solved = False
        elif self.nw.status in [2, 3, 99]:
            # in this case model is very likely corrupted!!
            # fix it by running a presolve using the stable solution
            self._solved = False
            self.nw.solve("design", init_only=True, init_path=self._stable_solution)

    def plot_Ts_diagram(self) -> plt.Figure:
        fig, ax = plt.subplots(1)
        return fig, ax

    def plot_logph_diagram(self) -> plt.Figure:
        fig, ax = plt.subplots(1)
        return fig, ax


## `CO2BoosterModel`

Full CO2 transcritical booster system: LT/MT evaporators, LT/MT compressors, receiver,
gas cooler, and an integrated heat-recovery heat exchanger.

In [ ]:
class CO2BoosterModel(ModelTemplate):

    def _parameter_lookup(self) -> dict:
        return {
            "lt_evaporator_load_kw": ["Components", "lt-evaporator", "Q"],
            "mt_evaporator_load_kw": ["Components", "mt-evaporator", "Q"],
            "lt_compressor_isentropic_eff": ["Components", "lt-compressor", "eta_s"],
            "mt_compressor_isentropic_eff": ["Components", "mt-compressor", "eta_s"],
            "lt_evaporator_pressure_bar": ["Connections", "c6", "p"],
            "mt_evaporator_pressure_bar": ["Connections", "c13", "p"],
            "receiver_pressure_bar": ["Connections", "c3", "p"],
            "gas_cooler_pressure_bar": ["Connections", "c10", "p"],
            "gas_cooler_outlet_temperature_C": ["Connections", "c1", "T"],
            "hrhx_water_inlet_temperature_C": ["Connections", "c11", "T"],
            "mt_compressor_power_w": ["Components", "mt-compressor", "P"],
            "lt_compressor_power_w": ["Components", "lt-compressor", "P"],
            "hrhx_heat_w": ["Components", "heat-recovery-heat-exchanger", "Q"]
        }

    def _create_network(self) -> None:

        super()._create_network()

        # === System components =========================================
        gas_cooler = SimpleHeatExchanger('gas-cooler')
        cc = CycleCloser('cc')
        hp_valve = Valve('high-pressure-valve')
        bp_valve = Valve('by-pass-valve')
        mt_exp_valve = Valve('mt-exp-valve')
        lt_exp_valve = Valve('lt-exp-valve')
        mt_evap = SimpleHeatExchanger('mt-evaporator')
        lt_evap = SimpleHeatExchanger('lt-evaporator')
        mt_comp = Compressor('mt-compressor')
        lt_comp = Compressor('lt-compressor')
        merge1 = Merge('mt-evaporator-outlet')
        merge2 = Merge('mt-compressor-suction')
        split = Splitter('receiver-liq-outlet')
        receiver = DropletSeparator('receiver')
        hrhx = SectionedHeatExchanger('heat-recovery-heat-exchanger')
        water_inlet = Source('water-inlet')
        water_outlet = Sink('water-outlet')

        # === Connections ===============================================
        w1 = Connection(water_inlet, 'out1', hrhx, 'in2')
        w2 = Connection(hrhx, 'out2', water_outlet, 'in1')

        c1 = Connection(gas_cooler, 'out1', cc, 'in1', 'c1')
        c2 = Connection(cc, 'out1', hp_valve, 'in1', 'c2')
        c3 = Connection(hp_valve, 'out1', receiver, 'in1', 'c3')
        c4 = Connection(receiver, 'out1', split, 'in1', 'c4')
        c5 = Connection(split, 'out1', lt_exp_valve, 'in1', 'c5')
        c6 = Connection(lt_exp_valve, 'out1', lt_evap, 'in1', 'c6')
        c7 = Connection(lt_evap, 'out1', lt_comp, 'in1', 'c7')
        c8 = Connection(lt_comp, 'out1', merge2, 'in1', 'c8')
        c9 = Connection(merge2, 'out1', mt_comp, 'in1', 'c9')
        c10 = Connection(mt_comp, 'out1', hrhx, 'in1', 'c10')
        c11 = Connection(hrhx, 'out1', gas_cooler, 'in1', 'c11')
        c12 = Connection(split, 'out2', mt_exp_valve, 'in1', 'c12')
        c13 = Connection(mt_exp_valve, 'out1', mt_evap, 'in1', 'c13')
        c14 = Connection(mt_evap, 'out1', merge1, 'in1', 'c14')
        c15 = Connection(merge1, 'out1', merge2, 'in2', 'c15')
        c16 = Connection(receiver, 'out2', bp_valve, 'in1', 'c16')
        c17 = Connection(bp_valve, 'out1', merge1, 'in2', 'c17')

        self.nw.add_conns(c1, c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12, c13, c14, c15, c16, c17, w1, w2)

        # === Component parametrisation ==================================
        gas_cooler.set_attr(dp=0)
        hrhx.set_attr(dp1=0, dp2=0)
        mt_comp.set_attr(eta_s=.8)
        lt_comp.set_attr(eta_s=.85)
        mt_evap.set_attr(pr=1, Q=75000)
        lt_evap.set_attr(pr=1, Q=25000)

        c1.set_attr(T=15, fluid={'R744': 1})
        c3.set_attr(p=32)
        c6.set_attr(p=13)
        c13.set_attr(p=26)
        c10.set_attr(p=57.3)
        c11.set_attr(T=35)
        lt_evap_T_sat = CP.PropsSI('T', 'P', c6.p.val * 1e5, 'Q', 0, 'R744')
        mt_evap_T_sat = CP.PropsSI('T', 'P', c13.p.val * 1e5, 'Q', 0, 'R744')
        c7.set_attr(td_dew=10)
        c14.set_attr(td_dew=10)

        w1.set_attr(T=30, p=3, fluid={'water': 1})
        w2.set_attr(T=55)

        # === Solve ======================================================
        self.nw.solve("design")

        hrhx.set_attr(td_pinch=5)
        c11.set_attr(T=None)
        self.nw.solve("design")

        self._stable_solution = "stable_solution.json"
        self.nw.save(self._stable_solution)

    def plot_Ts_diagram(self):
        fig, ax = super().plot_Ts_diagram()

        if "R744" not in DIAGRAMS:
            diagram = FluidPropertyDiagram("R744")
            diagram.set_unit_system(**{"T": "°C", "p": "bar"})
            diagram.set_isolines_subcritical(0, 70)
            diagram.calc_isolines()
            DIAGRAMS["R744"] = diagram

        diagram = DIAGRAMS["R744"]
        processes, points = get_plotting_data(self.nw, "c1")

        processes = {
            key: diagram.calc_individual_isoline(**value)
            for key, value in processes.items()
            if value is not None
        }

        diagram.draw_isolines(fig, ax, "Ts", 800, 2200, -40, 120)

        for label, values in processes.items():
            _ = ax.plot(values["s"], values["T"], label=label, color="tab:red")

        for label, point in points.items():
            _ = ax.scatter(point["s"], point["T"], label=label, color="tab:red")

        return fig, ax

    def plot_logph_diagram(self):
        fig, ax = super().plot_logph_diagram()

        diagram = FluidPropertyDiagram("R744")
        diagram.set_unit_system(**{"h": "J/kg", "p": "bar", "T": "°C"})

        diagram.set_isolines(
            T=np.arange(-50, 160, 10),
            p=np.array([]),
            h=np.array([]),
            s=np.array([]),
            v=np.array([]),
            Q=np.array([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
        )
        diagram.calc_isolines()

        DIAGRAMS["R744"] = diagram

        processes, points = get_plotting_data(self.nw, "c1")

        processes = {
            key: diagram.calc_individual_isoline(**value)
            for key, value in processes.items()
            if value is not None
        }

        diagram.draw_isolines(fig, ax, "logph", 100000, 800000, 6, 150)

        for label, values in processes.items():
            ax.plot(values["h"], values["p"], label=label, color="tab:red", linewidth=1)

        for label, point in points.items():
            ax.scatter(point["h"], point["p"], label=label, color="tab:red", s=25, zorder=5)

        return fig, ax


## Boundary conditions

Solves the CO2 booster model using the operating conditions of **Case Study 1** from the
book chapter.

Case Study 1 parameters (from the chapter):

| Parameter | Value |
|---|---|
| LT evaporator pressure | 13 bar (≈ -33 °C) |
| MT evaporator pressure | 26 bar (≈ -10 °C) |
| Receiver pressure | 32 bar |
| Gas cooler pressure | 57.3 bar |
| Gas cooler outlet temperature | 15 °C (5 °C subcooling) |
| LT cooling load | 25 kW |
| MT cooling load | 75 kW |
| LT compressor isentropic efficiency | 0.85 |
| MT compressor isentropic efficiency | 0.80 |

These already match the defaults hardcoded in `CO2BoosterModel._create_network()`, so
`solve_model(...)` is called here simply to make that traceability explicit and to show
how the same values would be passed in through the `ModelTemplate` parameter interface.

In [ ]:
# === Case Study 1 operating conditions (book chapter, Section 5.1) =========
case_study_1_inputs = {
    "lt_evaporator_pressure_bar": 13,
    "mt_evaporator_pressure_bar": 26,
    "receiver_pressure_bar": 32,
    "gas_cooler_pressure_bar": 57.3,
    "gas_cooler_outlet_temperature_C": 15,
    "lt_evaporator_load_kw": 25e3,   # Q_LT, book chapter uses kW; TESPy Q is in W
    "mt_evaporator_load_kw": 75e3,   # Q_MT
    "lt_compressor_isentropic_eff": 0.85,
    "mt_compressor_isentropic_eff": 0.80,
}

# === CO2 booster model ============================================================
model = CO2BoosterModel()
model.solve_model(**case_study_1_inputs)

# plot
model.plot_logph_diagram()
